In [1]:
from dotenv import load_dotenv
from tqdm.auto import tqdm
from ingest import load_faq_data, build_elastic_index
from rag_helper import RAGBase
from openai import OpenAI

In [2]:
load_dotenv()

True

In [3]:
documents = load_faq_data()

In [4]:
print(documents[0])

{'id': '0e38656cfb', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'How do I submit homework?', 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}


In [5]:
openai_client = OpenAI()

In [6]:
from elasticsearch import Elasticsearch

In [7]:
elastic_client = Elasticsearch("http://localhost:9200")

In [8]:
elastic_client.info()

ObjectApiResponse({'name': 'ff6a724889df', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'HLZmcn2TTWqAwnDz3eXKVQ', 'version': {'number': '8.4.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '42f05b9372a9a4a470db3b52817899b99a76ee73', 'build_date': '2022-10-04T07:17:24.662462378Z', 'build_snapshot': False, 'lucene_version': '9.3.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'})

In [8]:
index_name = "course-questions"

index_settings = {
    "settings": {
        "number_of_shards": 1,
        "number_of_replicas": 0
    },
    "mappings": {
        "properties": {
            #"text": {"type": "text"},
            "section": {"type": "text"},
            "question": {"type": "text"},
            "course": {"type": "keyword"} 
        }
    }
}

In [9]:
elastic_client.indices.create(
    index=index_name,
    body=index_settings
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'course-questions'})

In [12]:
for doc in tqdm(documents):
    elastic_client.index(
        index=index_name,
        document=doc
    )

  0%|          | 0/1208 [00:00<?, ?it/s]

In [4]:
query = "I just discovered the course. Can I join it ?"

In [14]:
search_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": {
                "multi_match": {
                    "query": query,
                    "fields": ["question^3", "section"],
                    "type": "best_fields"
                }
            },
            "filter": {
                "term": {
                    "course": "data-engineering-zoomcamp"
                }
            }
        }
    }
}

In [16]:
response = elastic_client.search(
    index=index_name,
    body=search_query
)

In [24]:
for resp in response['hits']['hits']:
    print(resp['_source'])

{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}
{'id': '068529125b', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I follow the course after it finishes?', 'answer': 'Yes, we will keep all the materials available, so you can follow the course at your own pace after it finishes.\n\nYou can also continue reviewing the homeworks and prepare for the next cohort. You can also start working on your final capstone project.'}
{'id': '33fc260cd8', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: What can I do

In [6]:
?build_elastic_index

Signature: build_elastic_index(documents)
Docstring: <no docstring>
File:      ~/Documents/Learning/llm-zoomcamp-code/ingest.py
Type:      function

In [6]:
index = build_elastic_index(documents)

  0%|          | 0/1208 [00:00<?, ?it/s]

In [10]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    course="data-engineering-zoomcamp"
)

In [11]:
assistant.rag(query="How do I get certificate")

TypeError: string indices must be integers, not 'str'